In [1]:
import sys
import importlib
sys.path.append("../src")

import pandas as pd
import numpy as np
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import predict
importlib.reload(predict)

df = pd.read_csv("../data/processed/features_v2.csv", parse_dates=["GAME_DATE"])
df = df.sort_values("GAME_DATE", kind="stable").reset_index(drop=True)
print(df.shape)

(11896, 29)


In [2]:
usable = df.dropna(subset=predict.FEATURES)
assert "HOME_PTS" not in predict.FEATURES and "AWAY_PTS" not in predict.FEATURES

final_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
final_model.fit(usable[predict.FEATURES], usable["HOME_WIN"])

joblib.dump(final_model, "../models/final_model.joblib")
print("Trained on", len(usable), "games")

Trained on 11896 games


In [3]:
state = predict.build_state(df)
joblib.dump(state, "../models/team_state.joblib")

print("Teams:", len(state["teams"]), "| last game in data:", state["last_date"].date())
pd.DataFrame.from_dict(state["teams"], orient="index").sort_values("elo", ascending=False).head(5)

Teams: 30 | last game in data: 2025-04-13


,elo,pts_roll10,allowed_roll10,win_roll10,last_game
OKC,1751.100198,125.2,112.9,0.8,2025-04-13
BOS,1680.697733,113.8,102.6,0.8,2025-04-13
CLE,1680.278019,119.7,116.4,0.6,2025-04-13
MIN,1653.818420,118.6,110.1,0.8,2025-04-13
LAC,1653.394116,119.4,106.0,0.9,2025-04-13


In [4]:
last_season_start = df.index[df["SEASON"] == df["SEASON"].iloc[-1]][0]

for i in (len(df) - 1, len(df) - 500, last_season_start):
    g = df.loc[i]
    state_before = predict.build_state(df.iloc[:i])
    rebuilt = predict.make_features(state_before, g["HOME_TEAM"], g["AWAY_TEAM"], g["GAME_DATE"])
    stored = g[predict.FEATURES].astype(float).to_numpy()
    ok = np.allclose(rebuilt.iloc[0].to_numpy(dtype=float), stored, atol=1e-6)
    print(f"{g['GAME_DATE'].date()} {g['AWAY_TEAM']} @ {g['HOME_TEAM']}: match = {ok}")
    if not ok:
        print(pd.DataFrame({"rebuilt": rebuilt.iloc[0].to_numpy(dtype=float), "stored": stored},
                           index=predict.FEATURES))

2025-04-13 DAL @ MEM: match = True
2025-02-03 SAC @ MIN: match = True
2024-10-22 MIN @ LAL: match = True


In [5]:
model, state = predict.load_artifacts("../models")

elo = {t: s["elo"] for t, s in state["teams"].items()}
best = max(elo, key=elo.get)
worst = min(elo, key=elo.get)
date = state["last_date"] + pd.Timedelta(days=3)

p1 = predict.predict_game(model, state, best, worst, date)
p2 = predict.predict_game(model, state, worst, best, date)
print(f"{best} (home) vs {worst}: {p1:.3f}")
print(f"{worst} (home) vs {best}: {p2:.3f}")
print(f"Sum: {p1 + p2:.3f}")

OKC (home) vs WAS: 0.962
WAS (home) vs OKC: 0.056
Sum: 1.018
